In [1]:
import h5py, numpy as np
DATA = "data"

with h5py.File(f"{DATA}/vg150/VG-SGG.h5", "r") as f:
    boxes = f['boxes_1024'][:]
    boxes_512 = f['boxes_512'][:]
    first_b = f['img_to_first_box'][:]
    last_b = f['img_to_last_box'][:]
    splits = f['split'][:]

w, h = boxes[:,2], boxes[:,3]
xc, yc = boxes[:,0], boxes[:,1]
print(f"{len(boxes)} boxes loaded")


1145398 boxes loaded


In [2]:
# first 5 boxes raw
for i in range(5):
    print(f"[{i}] {boxes[i]} label=...")
xc0, yc0, w0, h0 = boxes[0]
x1, y1 = int(xc0 - w0/2), int(yc0 - h0/2)
x2, y2 = int(xc0 + w0/2), int(yc0 + h0/2)
print(f"box 0 converted: ({x1},{y1}) -> ({x2},{y2}) {x2-x1}x{y2-y1}")


[0] [ 511  356 1023  713] label=...
[1] [560 578 925 372] label=...
[2] [142 344 285 689] label=...
[3] [790 525 460 331] label=...
[4] [587 311  99 465] label=...
box 0 converted: (0,0) -> (1022,712) 1022x712


In [3]:
# bad box check
zero_w, zero_h = (w == 0).sum(), (h == 0).sum()
neg_w, neg_h = (w < 0).sum(), (h < 0).sum()
print(f"zero w: {zero_w}, zero h: {zero_h}")
print(f"negative w: {neg_w}, negative h: {neg_h}")


zero w: 0, zero h: 0
negative w: 0, negative h: 0


In [4]:
# aspect ratio
ratio = np.where(h > 0, w / h, 0)
print(f"tall (<0.5): {(ratio < 0.5).sum()} ({(ratio < 0.5).sum()/len(boxes)*100:.1f}%)")
print(f"square (0.8-1.2): {((ratio >= 0.8) & (ratio <= 1.2)).sum()}")
print(f"wide (>2): {(ratio > 2).sum()}")
print(f"min={ratio.min():.2f} max={ratio.max():.2f}")


tall (<0.5): 197631 (17.3%)
square (0.8-1.2): 269558
wide (>2): 193813
min=0.00 max=340.67


In [5]:
# box area percentiles
areas = w * h
for pct in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    v = np.percentile(areas, pct)
    print(f"p{pct:2d}: {v:8.0f}")


p 1:     1599
p 5:     2146
p10:     2916
p25:     6540
p50:    21879
p75:    81920
p90:   241032
p95:   390944
p99:   694960


In [6]:
# objects per image by split
for split_val, name in [(0,"train"),(2,"test")]:
    mask = splits == split_val
    counts = last_b[mask] - first_b[mask] + 1
    valid = counts[counts > 0]
    print(f"{name}: {len(valid)} imgs, {valid.sum()} boxes, {valid.mean():.1f} obj/img")
    print(f"  1 obj: {(valid==1).sum()}, 2-5: {((valid>=2)&(valid<=5)).sum()}, 10+: {(valid>=10).sum()}")


train: 75651 imgs, 795174 boxes, 10.5 obj/img
  1 obj: 3275, 2-5: 12653, 10+: 38861
test: 32422 imgs, 352883 boxes, 10.9 obj/img
  1 obj: 1294, 2-5: 6103, 10+: 16769


In [7]:
# compare 1024 vs 512 scale
for i in range(3):
    print(f"{boxes[i]} vs {boxes_512[i]}  ratio={boxes[i]/boxes_512[i]}")


[ 511  356 1023  713] vs [256 178 512 357]  ratio=[1.99609375 2.         1.99804688 1.99719888]
[560 578 925 372] vs [280 289 463 186]  ratio=[2.         2.         1.99784017 2.        ]
[142 344 285 689] vs [ 71 172 143 345]  ratio=[2.         2.         1.99300699 1.99710145]
